# Colab layout fine-tune — DocLayout-YOLO on our documents (Track A, §2.43)

**Run this on Colab, not locally** — training `imgsz=1024` freezes the 24GB Mac (it already hosts Surya+MLX). A **free T4** does it in ~10–20 min.

Fine-tunes the DocStructBench-pretrained DocLayout-YOLO (the §2.24 off-the-shelf loser) on our human-corrected `layout_v1_corrected` set (84/9/9). Starting from a model that already knows document layout — **not** training a detector from scratch on 84 images.

**Output:** `khmer_layout_best.onnx` (auto-downloaded), with the `character` class-list metadata `rapid_layout` needs already injected. Drop it anywhere on the Mac and run the gates with `KHMER_LAYOUT_WEIGHTS=/path/to/khmer_layout_best.onnx` — no local post-processing.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# doclayout-yolo is the research fork; onnx/onnxslim/onnxscript are for the export step.
!pip install -q doclayout-yolo huggingface-hub onnx onnxslim onnxscript onnxruntime

In [ ]:
# Dataset: zip eval/datasets/layout_v1_corrected/ locally and upload it here.
#   (repo root)  cd eval/datasets && zip -r layout_v1_corrected.zip layout_v1_corrected
from google.colab import files
import zipfile, os
up = files.upload()  # pick layout_v1_corrected.zip
zname = next(k for k in up if k.endswith('.zip'))
with zipfile.ZipFile(zname) as z:
    z.extractall('data')
# Find the dir that actually contains data.yaml (zip may or may not nest a folder).
DATASET = next(os.path.dirname(os.path.join(r, 'data.yaml'))
               for r, _, fs in os.walk('data') if 'data.yaml' in fs)
print('dataset:', DATASET)

In [ ]:
# Roboflow's data.yaml uses paths relative to itself; rewrite them absolute so the
# trainer finds the splits regardless of CWD. (Mirrors experiments/layout_yolo/finetune_doclayout.py.)
import os
names = next(l for l in open(os.path.join(DATASET, 'data.yaml')) if l.startswith('names:'))
with open('data_local.yaml', 'w') as f:
    f.write(f"train: {os.path.abspath(DATASET)}/train/images\n")
    f.write(f"val: {os.path.abspath(DATASET)}/valid/images\n")
    f.write(f"test: {os.path.abspath(DATASET)}/test/images\n")
    f.write(f"nc: 5\n{names}")
print(open('data_local.yaml').read())

In [ ]:
# Fine-tune from the DocStructBench-pretrained checkpoint. seed=0 + deterministic
# so the run is reproducible for the report. imgsz=1024 matches the pretrained res.
from huggingface_hub import hf_hub_download
from doclayout_yolo import YOLOv10

ckpt = hf_hub_download(repo_id='juliozhao/DocLayout-YOLO-DocStructBench',
                       filename='doclayout_yolo_docstructbench_imgsz1024.pt')
model = YOLOv10(ckpt)
# nc mismatch (10 DocStructBench classes → our 5) is handled by the trainer: the
# detection head is rebuilt from data.yaml, the backbone transfers.
model.train(data='data_local.yaml', epochs=100, imgsz=1024, patience=20,
            seed=0, deterministic=True, batch=8, device=0,
            project='runs', name='khmer_layout')

In [ ]:
# Gate 1 (probe): test-split mAP@50 per class. Want Table ≳ 0.9. Read straight off.
metrics = model.val(data='data_local.yaml', split='test', device=0)
print(f'mAP50 (all): {metrics.box.map50:.3f}')
for i, name in metrics.names.items():
    if i < len(metrics.box.ap50):
        print(f'  {name:16s} mAP50={metrics.box.ap50[i]:.3f}')
print('best weights:', model.trainer.best)

In [ ]:
# Export best.pt → ONNX and inject the `character` class-list metadata that
# rapid_layout reads (an exported model has no such key → KeyError: 'character').
# This mirrors experiments/layout_yolo/export_onnx.py so the Mac needs zero post-processing.
best = str(model.trainer.best)
onnx_out = YOLOv10(best).export(format='onnx', imgsz=1024, opset=13)

import onnx
cls = [n.strip().strip("'\"") for n in names.split('names:', 1)[1].strip().strip('[]').split(',') if n.strip()]
m = onnx.load(onnx_out)
for i, p in enumerate(list(m.metadata_props)):
    if p.key == 'character':
        del m.metadata_props[i]; break
e = m.metadata_props.add(); e.key = 'character'; e.value = '\n'.join(cls)
onnx.save(m, 'khmer_layout_best.onnx')
print('classes embedded:', cls)

In [ ]:
# Download the drop-in ONNX. On the Mac:
#   KHMER_LAYOUT_WEIGHTS=/path/khmer_layout_best.onnx uv run python experiments/layout_yolo/gate.py
from google.colab import files
files.download('khmer_layout_best.onnx')